<a href="https://colab.research.google.com/github/Sahil-Sahu3/Machine-Learning/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sahil-Sahu3/Machine-Learning/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

The most appropriate machine learning task type for identifying and prioritizing web pages for content writers to fix is Ranking. Here's the justification:

Prioritization is Key: The core of the problem is not just identifying pages that need fixing, but prioritizing them. Content writers have limited time, so an 'automated supervisor' needs to present a list of pages ordered by their impact or urgency for improvement. Ranking models are specifically designed to order items based on relevance or a predicted score.

Beyond Binary Classification: While a classification model could identify 'needs fix' vs. 'doesn't need fix', it wouldn't inherently provide the necessary prioritization. A page identified as 'needs fix' might be low priority, while another might be critical. Ranking addresses this nuance directly.

Scoring for Relative Importance: Ranking models often work by assigning a score to each item (page, in this case) and then ordering them by this score. This score can be a composite of various factors indicating a page's potential for improvement, its current poor performance, or its strategic importance.

Dynamic and Adaptive: A ranking approach allows for a dynamic system that can adapt to changing user behavior, search trends, and content performance metrics without needing constant rule adjustments. The model learns the optimal weighting of various signals to produce a prioritized list.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

The decision to derive the `page_improvement_score` is practical and aligns with the goal of creating an 'automated supervisor' system:

1.  **Leveraging Existing Data**: We have millions of rows of search data (e.g., Google Search Console data, analytics data). This data contains rich signals that, when combined, can indicate page health and potential for improvement.
2.  **Operational Feasibility**: Creating a robust, continuously updated ground truth based on observed outcomes for millions of pages is not feasible. Defining rules allows us to generate a target variable that can be systematically applied across the entire dataset.
3.  **Incorporating Domain Expertise**: The rules and heuristics used to define `page_improvement_score` can incorporate the domain expertise of SEO specialists and content strategists, translating their qualitative understanding of 'what makes a page bad' into quantifiable metrics.
4.  **Signals for Derivation**: The `page_improvement_score` would be a weighted combination of factors such as:
    *   **Low Click-Through Rate (CTR)** for high-impression queries.
    *   **High Bounce Rate** or **Low Dwell Time** (from analytics).
    *   **Declining Organic Search Impressions/Clicks** over a period.
    *   **Low Average Position** for important keywords.
    *   **High keyword cannibalization** (multiple pages ranking for the same target keyword).
    *   **Missing or Thin Content** (inferred from page word count or structure).
    *   **High number of competing pages** for relevant keywords.

This derived score provides a quantifiable proxy for the 'fix priority' that the ranking model will learn to predict, guiding content writers to the most impactful tasks.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

For a ranking model, **Normalized Discounted Cumulative Gain (NDCG)** is an ideal primary success metric. NDCG is commonly used to measure the quality of a ranked list, taking into account both the relevance of the items and their position in the list.

### Justification:

1.  **Relevance and Position**: NDCG rewards highly relevant items (pages with a high `page_improvement_score`) being placed higher in the ranked list. This is crucial because content writers have limited time, and we want them to work on the most impactful pages first.
2.  **Graded Relevance**: Unlike binary metrics (e.g., precision@k), NDCG can incorporate graded relevance. Since our target variable is a continuous `page_improvement_score`, NDCG naturally handles this by using the actual score (or a transformation of it) as the 'gain' for each item.
3.  **Discounting Positional Value**: The 'Discounted' part means that relevant items at lower positions are penalized. This aligns with our goal: a highly impactful page is much more valuable if it's at the top of the content writer's queue than at the bottom.
4.  **Normalization**: The 'Normalized' part means the score is independent of the number of items in the list, allowing for comparison across different ranking lists or models. It scales the score to a value between 0 and 1, where 1 is the perfect ranking.
5.  **Business Impact**: By optimizing for NDCG, we aim to ensure that content writers consistently address the pages that offer the highest potential for improvement (e.g., increased organic traffic, conversions) first, thereby maximizing the business impact of their efforts.

### What Constitutes a 'Good' Outcome?

An **NDCG@k score of 0.85 or higher** would indicate a 'good' outcome for the model. Here's why:

*   **NDCG Scale**: NDCG values range from 0 to 1. A score of 1 represents a perfect ranking where all items are ordered by their true relevance (our derived `page_improvement_score`).
*   **Practical Threshold**: Achieving a perfect score (1.0) is often unrealistic in complex real-world scenarios. A score of 0.85 suggests that the model is performing very well, placing the most important pages (with high `page_improvement_score`) consistently near the top of the recommended list.
*   **Relative Improvement**: While 0.85 is a target, the true success will also be measured by **relative improvement** over a baseline (e.g., a rule-based ranking or random ranking). If the ML model consistently achieves an NDCG score significantly higher than current ad-hoc methods or simple heuristics, it would demonstrate its value.
*   **`@k` Consideration**: The `@k` in NDCG@k refers to evaluating the ranking quality for the top `k` items. For content writers, `k` might be set to a reasonable number of pages they can realistically address in a given period (e.g., `@10`, `@20`, `@50`). The target of 0.85 would apply to the relevant `k` for which the model's recommendations are most acted upon.

Ultimately, the 'goodness' will also be validated by downstream business metrics, such as the observed increase in organic traffic or conversions from pages that were improved based on the model's high-priority recommendations.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

### Representation as a Single Row in a Dataframe:

Each row in our hypothetical dataframe would represent one `Page-Month` and would contain the following types of features, aggregated or derived from millions of rows of raw search data (e.g., Google Search Console, Google Analytics, internal site data):

| Feature Name                 | Data Type    | Description                                                                                                                                                             | Derivation/Source                                                                                                                                  |
| :--------------------------- | :----------- | :---------------------------------------------------------------------------------------------------------------------------------------------------------------------- | :------------------------------------------------------------------------------------------------------------------------------------------------- |
| `page_id`                    | String       | Unique identifier for the web page (e.g., canonical URL).                                                                                                               | Derived from raw URL data.                                                                                                                         |
| `month`                      | Date         | The month for which the data is aggregated.                                                                                                                             | Derived from timestamp of search data.                                                                                                             |
| `total_impressions_month`    | Integer      | Total search impressions for the page in the given month.                                                                                                               | Sum of impressions from GSC data for `page_id` in `month`.                                                                                         |
| `total_clicks_month`         | Integer      | Total search clicks for the page in the given month.                                                                                                                    | Sum of clicks from GSC data for `page_id` in `month`.                                                                                              |
| `avg_ctr_month`              | Float        | Average Click-Through Rate for the page in the given month.                                                                                                             | `total_clicks_month` / `total_impressions_month`.                                                                                                  |
| `avg_position_month`         | Float        | Average search ranking position for the page in the given month.                                                                                                        | Average of daily average positions from GSC data for `page_id` in `month`.                                                                         |
| `bounce_rate_month`          | Float        | Average bounce rate for the page from organic search in the given month.                                                                                                | From Google Analytics for `page_id` in `month`.                                                                                                    |
| `avg_time_on_page_month`     | Float        | Average time users spent on the page from organic search in the given month.                                                                                            | From Google Analytics for `page_id` in `month`.                                                                                                    |
| `word_count`                 | Integer      | Number of words on the page.                                                                                                                                            | Scraped from the page content (static feature, might be updated periodically).                                                                     |
| `num_internal_links`         | Integer      | Number of internal links pointing to this page.                                                                                                                         | From internal site crawl data (static feature, might be updated periodically).                                                                     |
| `num_external_links`         | Integer      | Number of external links pointing to this page.                                                                                                                         | From SEO backlink tools (static feature, might be updated periodically).                                                                           |
| `traffic_trend_3m`           | Float        | % change in total organic clicks over the last 3 months.                                                                                                                | (Clicks this month - Clicks 3 months ago) / Clicks 3 months ago.                                                                                   |
| `ctr_vs_expected`            | Float        | Difference between actual CTR and an expected CTR for its average position (e.g., from an industry benchmark curve).                                                      | Derived from GSC data and a benchmark curve.                                                                                                       |
| `has_h1`                     | Boolean      | Whether the page has an H1 tag.                                                                                                                                         | Scraped from page content.                                                                                                                         |
| `has_meta_description`       | Boolean      | Whether the page has a meta description.                                                                                                                                | Scraped from page content.                                                                                                                         |
| `top_keywords_coverage`      | Integer      | Number of high-volume keywords for which the page ranks but isn't in the top 3.                                                                                          | Derived from GSC queries data.                                                                                                                     |
| `page_improvement_score`     | Float        | **Target Variable**: Derived score indicating potential for improvement.                                                                                                | Calculated based on predefined rules using a combination of the above features and their trends (e.g., low CTR for high impressions, declining traffic, poor on-page SEO). |

### Feature Derivation from Raw Search Data:

*   **Aggregation**: Metrics like `total_impressions_month`, `total_clicks_month`, `avg_position_month`, `bounce_rate_month`, `avg_time_on_page_month` would be aggregated from daily or raw event-level data (e.g., Google Search Console queries, Google Analytics page views) for each unique `page_id` within a given `month`.
*   **Calculation**: `avg_ctr_month` is calculated from aggregated clicks and impressions. `traffic_trend_3m` involves comparing aggregated monthly clicks over a period.
*   **Static/Periodically Updated Features**: Features like `word_count`, `num_internal_links`, `has_h1`, `has_meta_description` might be obtained from a site crawl or content management system (CMS) at regular intervals (e.g., weekly or monthly) and joined with the time-series search data.
*   **Derived Heuristics**: `ctr_vs_expected` would involve a more complex calculation comparing the page's actual CTR to a benchmark based on its average ranking position. `top_keywords_coverage` would involve analyzing keyword performance data.
*   **Target Variable**: The `page_improvement_score` would be a composite score generated using predefined rules and weights applied to a combination of these features and their historical trends, embodying the domain expertise of what makes a page a priority for fixing.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

1.  **Over-simplification of Nuance**: Content performance is rarely dictated by a single metric or a simple threshold. A page might have low CTR but high impressions (suggesting a title issue), or high bounce rate with declining organic traffic (suggesting content quality issues). Rule-based systems struggle to combine these signals effectively without becoming unwieldy.
2.  **Brittleness and Maintenance Burden**: As search algorithms change, user behavior evolves, and new content is added, fixed rules quickly become outdated or suboptimal. Maintaining and updating a complex web of `if-then-else` statements across millions of pages and hundreds of potential signals would be a full-time job for a team of experts.
3.  **Inability to Discover Hidden Patterns**: Rule-based systems are limited to explicitly defined conditions. They cannot identify subtle, non-linear relationships or emergent patterns in the data that indicate a page needs fixing or has high improvement potential. For instance, a combination of slight CTR drop, slight position drop, and increased competition might be a strong signal, but hard to capture with simple rules.
4.  **Lack of Prioritization Depth**: While rules can identify pages that meet certain criteria (e.g., "CTR < 5% AND impressions > 1000"), they struggle to provide a nuanced ranking of *how much* a page needs fixing, or which pages offer the highest return on a content writer's investment. A page with a 4% CTR might be a higher priority than one with a 3% CTR if the former has significantly higher impressions and competitive keywords.
5.  **Difficulty in Attributing "Why"**: While rules can state *what* condition was met, they often fail to explain the complex interplay of factors leading to a low-performing page or provide a comprehensive "why" that is actionable for a content writer. For example, a rule might flag "low CTR," but not easily infer if it's due to poor meta description, irrelevant search intent, or too many ads on the page.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Final Task

### Subtask:
Summarize the complete framing of the machine learning task, including the type, target, metric, unit of analysis, and rationale for using ML to create a prioritized list of pages for content writers.

## Summary: Framing the 'Automated Supervisor' ML Task

**Problem:** To build an 'automated supervisor' system that identifies and prioritizes web pages for content writers to fix, based on millions of rows of search data, and provides reasons for intervention.

**1. ML Task Type: Ranking**
*   **Why:** The core objective is to prioritize pages for content writers. Classification only tells us *if* a page needs fixing, but ranking provides an ordered list based on impact or urgency, allowing content writers to focus on the most valuable tasks first.

**2. Target Variable: `page_improvement_score`**
*   **What:** A continuous numerical score representing the estimated potential impact or urgency of fixing a given page.
*   **Origin:** Derived from **defined rules and heuristics** rather than direct observed outcomes. This composite score combines various signals (e.g., low CTR for high impressions, declining traffic, poor on-page SEO) to quantify potential for improvement.

**3. Success Metric: Normalized Discounted Cumulative Gain (NDCG@k)**
*   **Why:** NDCG is ideal for ranking tasks as it considers both the relevance of items (our `page_improvement_score`) and their position in the ranked list, rewarding highly relevant items placed higher. It handles graded relevance and normalizes scores for comparison.
*   **'Good' Outcome:** An NDCG@k score of **0.85 or higher** would indicate excellent model performance, effectively guiding content writers to the most impactful pages.

**4. Unit of Analysis: `Page-Month`**
*   **What:** Each row in the dataset represents a specific web page's performance and characteristics over a single calendar month.
*   **Why:** This granularity captures trends and changes, provides actionable insights for content writers, allows for appropriate feature aggregation from raw search data, and aligns well with the derivation of the `page_improvement_score`.
*   **Features:** Includes aggregated metrics like `total_impressions_month`, `total_clicks_month`, `avg_ctr_month`, `avg_position_month`, `bounce_rate_month`, `avg_time_on_page_month`, as well as derived features like `traffic_trend_3m`, `ctr_vs_expected`, and static page characteristics (`word_count`, `has_h1`).

**5. Why ML Beats a Fixed Rule:**
*   **Complexity & Nuance:** Content performance is driven by complex, often non-linear interactions between numerous signals that fixed rules struggle to capture effectively.
*   **Adaptability:** ML models can learn and adapt to changing search algorithms, user behavior, and content trends without constant manual rule updates.
*   **Deep Prioritization:** ML excels at assigning nuanced `page_improvement_score` values, enabling granular prioritization beyond simple thresholds.
*   **Pattern Discovery:** ML can uncover subtle, hidden patterns that indicate improvement potential, which are impossible to codify with explicit rules.
*   **Scalability & Actionable Insights:** ML efficiently processes millions of data rows to provide data-driven
